# Smart Keyboard: train and evaluate on Google Colab

Runs the whole pipeline of the repository on a free GPU. Before you start:
**Runtime → Change runtime type → T4 GPU**.

Every step saves its output to Google Drive, so if Colab disconnects you can re-run the setup cells and
continue from the step that was interrupted.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/Enes-Ozbek/smart-keyboard.git'   # <- change if your repo has another name
SAVE = '/content/drive/MyDrive/smart-keyboard'           # artifacts and results live here

In [ ]:
import os
if not os.path.exists('/content/smart-keyboard'):
    !git clone $REPO_URL /content/smart-keyboard
%cd /content/smart-keyboard
!git pull
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# every command below uses the same folders on Google Drive
ARGS = f'--artifacts-dir {SAVE}/artifacts --results-dir {SAVE}/results'

## 2. Data and the classical baseline (area B)

In [ ]:
!python -m keyboard.cli prepare $ARGS

In [ ]:
!python -m keyboard.cli bpe $ARGS

In [ ]:
!python -m keyboard.cli ngram $ARGS

## 3. Feedforward neural language model (area A)

In [ ]:
!python -m keyboard.cli ffnn $ARGS

## 4. Fine-tune GPT-2 small (area C)

In [ ]:
!python -m keyboard.cli gpt2 $ARGS

## 5. INT8 quantization (area D)

In [ ]:
!python -m keyboard.cli quantize $ARGS

## 6. Evaluate all models on the test split

Perplexity of the FP32 models uses the GPU; top-3 accuracy and latency run on the CPU of this Colab machine
(written under the table). This step takes a while because every model makes 1,000 timed predictions.
If the runtime was reset, run `prepare` again first (it recreates `data/`).

In [ ]:
!python -m keyboard.cli evaluate $ARGS
print(open(f'{SAVE}/results/results.md').read())

## 7. Upload the weights to the Hugging Face Hub (R7: no weights in git)

Create a write token at https://huggingface.co/settings/tokens first.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_REPO = 'USER/smart-keyboard-models'    # <- your Hugging Face user name
!python -m keyboard.cli upload $ARGS --models-repo $HF_REPO

## 8. Try the app (public link)

In [ ]:
!python app.py --share --artifacts-dir $SAVE/artifacts